In [ ]:
"""
Scenario 3 — Threshold Optimisation Dashboard
Finds the noise-vs-sensitivity sweet spot for production deployment.
"""

import os
os.environ["QT_QPA_PLATFORM"] = "xcb"

import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BASE = os.path.expanduser("~/itek/senstech-simulation/data")
DET = "medipix3_detector"

thresholds = [300, 500, 700, 1000, 1500, 2000]
n_events = 10000

results = []
hit_maps = []
charge_specs = []

for T in thresholds:
    path = os.path.join(BASE, f"s3_cdte_thresh{T}e.root")
    f = uproot.open(path)
    pc = f[f"DetectorHistogrammer/{DET}/charge/pixel_charge;1"].to_numpy()
    hm = f[f"DetectorHistogrammer/{DET}/hit_map;1"].to_numpy()
    cs = f[f"DetectorHistogrammer/{DET}/cluster_size/cluster_size;1"].to_numpy()

    hits = int(pc[0].sum())
    mean_q = np.average(pc[1][:-1], weights=pc[0] + 1e-9)
    mean_cs = np.average(cs[1][:-1], weights=cs[0] + 1e-9)

    results.append({"threshold": T, "hits": hits,
                    "efficiency": (hits / n_events) * 100,
                    "mean_charge": mean_q, "mean_cluster": mean_cs})
    hit_maps.append(hm[0])
    charge_specs.append(pc)

NAVY, WHITE, LGRAY = "#0B1F3A", "#F8FAFC", "#CBD5E1"
TEAL, AMBER, RED = "#0D9488", "#D97706", "#DC2626"

def style_ax(ax, title):
    ax.set_facecolor("#0D2040")
    ax.tick_params(colors=LGRAY, labelsize=9)
    ax.xaxis.label.set_color(LGRAY); ax.yaxis.label.set_color(LGRAY)
    ax.title.set_color(WHITE)
    ax.set_title(title, fontweight="bold", fontsize=11, pad=10)
    for s in ax.spines.values(): s.set_edgecolor("#1E3A5F")
    ax.grid(True, alpha=0.15, color=LGRAY)

fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor(NAVY)
gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.55, wspace=0.4,
                       left=0.06, right=0.97, top=0.92, bottom=0.06,
                       height_ratios=[1, 0.9, 0.9])

# Main curve
ax1 = fig.add_subplot(gs[0, :3])
style_ax(ax1, "Detection Efficiency vs Threshold")
effs = [r["efficiency"] for r in results]
ax1.plot(thresholds, effs, color=TEAL, linewidth=2.5, marker="o", markersize=10,
         markerfacecolor=WHITE, markeredgewidth=2)
# Mark noise floor zone
ax1.axvspan(0, 240, alpha=0.2, color=RED, label="Noise zone (< 3× ENC)")
ax1.axvline(x=700, color=AMBER, linestyle="--", linewidth=2,
            label="Datasheet minimum (700e)")
for r in results:
    ax1.annotate(f"{r['efficiency']:.0f}%", xy=(r["threshold"], r["efficiency"]),
                 xytext=(0, 12), textcoords="offset points", ha="center",
                 color=WHITE, fontsize=10, fontweight="bold")
ax1.set_xlabel("Threshold (electrons)")
ax1.set_ylabel("Detection Efficiency (%)")
ax1.set_xticks(thresholds)
ax1.legend(loc="upper right", fontsize=9, labelcolor=WHITE,
           facecolor="#0D2040", edgecolor="#1E3A5F")

# Bar chart of hits
ax2 = fig.add_subplot(gs[0, 3:])
style_ax(ax2, "Total Hits per Threshold")
colors = ["#DC2626", "#EA580C", "#0D9488", "#0369A1", "#7C3AED", "#6B21A8"]
bars = ax2.bar([str(t) + "e" for t in thresholds],
               [r["hits"] for r in results], color=colors, alpha=0.85,
               edgecolor=WHITE, linewidth=1)
for bar, r in zip(bars, results):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f"{r['hits']:,}", ha="center", color=WHITE,
             fontsize=10, fontweight="bold")
ax2.set_xlabel("Threshold setting")
ax2.set_ylabel("Total Pixel Hits")

# Hit maps
for i, T in enumerate(thresholds):
    ax = fig.add_subplot(gs[1, i])
    style_ax(ax, f"{T}e threshold")
    hm = hit_maps[i]
    c = hm.shape[0] // 2; cr = 60
    ax.imshow(hm[c-cr:c+cr, c-cr:c+cr], cmap="hot", origin="lower",
              aspect="equal", vmax=max(h.max() for h in hit_maps))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor(colors[i]); s.set_linewidth(3)

# Charge spectra
for i, T in enumerate(thresholds):
    ax = fig.add_subplot(gs[2, i])
    style_ax(ax, f"{T}e threshold")
    vals = charge_specs[i]
    m = vals[0] > 0
    ax.bar(vals[1][:-1][m], vals[0][m], width=np.diff(vals[1])[m],
           color=colors[i], alpha=0.85)
    ax.set_xlabel("Charge (e)", fontsize=8)
    ax.set_ylabel("Counts", fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle("Threshold Optimisation — CdTe on Medipix3 (Scenario 3)\n"
             "Finding the noise-vs-sensitivity sweet spot  ·  University of Surrey × Sens Tech",
             fontsize=14, fontweight="bold", color=WHITE, y=0.98)

out = os.path.expanduser("~/itek/senstech-simulation/analysis/notebooks/s3_threshold_dashboard.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

print("\n── Summary ───────────────────────────────────")
for r in results:
    print(f"  {r['threshold']:>5}e → {r['hits']:>6,} hits | {r['efficiency']:>5.1f}% efficiency")